# Research notebook
Run the bootstrap cell first. Review the experiment parameters and data paths before executing the remaining cells. Outputs are intentionally cleared for version control.


In [ ]:
from pathlib import Path
import os
import sys
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
os.chdir(project_root)
sys.path.insert(0, str(project_root / "src"))
Path("runs/notebooks").mkdir(parents=True, exist_ok=True)


In [ ]:
"""!conda install -y -c rapidsai -c conda-forge -c nvidia \
    cudf=26.04 cupy pandas matplotlib seaborn tqdm jinja2 ipython \
    python=3.11 cuda-version=12
"""

In [ ]:
# ============================================================
# LOB Similarity Matrices - Pearson / Spearman / NMI
# Multi-Lag + Multi-Bin Experiment
# NMI optimized with relative-lag approximation
# ============================================================

import os
import gc
import warnings
warnings.filterwarnings("ignore")

import cudf
import cupy as cp
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns


# ============================================================
# CONFIG
# ============================================================

# Path to the LOBSTER CSV file
CSV_PATH = "LOBSTER_SampleFile_AAPL_2012-06-21_10/AAPL_2012-06-21_34200000_57600000_orderbook_10.csv"

NUM_LEVELS = 10
LAG_LIST = [150]
N_BINS_LIST = [2000]

USE_FLOAT32 = True
NROWS = None

SAVE_CSV = True
PLOT_HEATMAPS = True

MAX_FEATURES_FOR_NMI = np.inf
MAX_FEATURES_FOR_PLOT = np.inf
MAX_FEATURES_HARD_LIMIT = np.inf

# use lag-relative approximation for NMI
USE_RELATIVE_LAG_NMI = True

RANDOM_SEED = 42

In [ ]:
# ============================================================
# UTILS - GENERAL
# ============================================================

def clear_gpu_memory():
    try:
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    except Exception:
        pass
    gc.collect()


def offdiag_mean(df: pd.DataFrame) -> float:
    arr = df.values
    idx = np.triu_indices_from(arr, k=1)
    vals = arr[idx]
    vals = vals[~np.isnan(vals)]
    if vals.size == 0:
        return float("nan")
    return float(vals.mean())


def quick_matrix_checks(df: pd.DataFrame, name: str):
    arr = df.values
    symmetric_error = np.nanmax(np.abs(arr - arr.T))
    diag_mean = np.nanmean(np.diag(arr))

    print(f"{name} diagonal mean      : {diag_mean:.6f}")
    print(f"{name} max symmetry error: {symmetric_error:.6e}")


def plot_heatmap(df: pd.DataFrame, title: str, cmap: str = "YlGnBu", vmin=None, vmax=None):
    plt.figure(figsize=(20, 16))
    sns.heatmap(
        df,
        annot=False,
        cmap=cmap,
        cbar_kws={"label": title},
        linewidths=0,
        square=True,
        vmin=vmin,
        vmax=vmax
    )
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(fontsize=8)
    plt.title(title, fontsize=16)
    plt.tight_layout()
    plt.show()


def parse_feature_name(name: str):
    parts = name.split("_")
    side = parts[0]
    level = int(parts[1])
    lag = int(parts[3])
    base = f"{side}_{level}"
    return side, level, lag, base


# ============================================================
# UTILS - NMI
# ============================================================

def gpu_entropy_from_counts(counts: cp.ndarray, n_samples: int) -> cp.float32:
    probs = counts[counts > 0].astype(cp.float32) / cp.float32(n_samples)
    if probs.size == 0:
        return cp.float32(0.0)
    return -cp.sum(probs * cp.log(probs))


def gpu_nmi_discrete(x: cp.ndarray, y: cp.ndarray, n_bins: int) -> float:
    n = x.size
    if n == 0:
        return 0.0

    cx = cp.bincount(x, minlength=n_bins)
    cy = cp.bincount(y, minlength=n_bins)

    joint_idx = x * n_bins + y
    cxy = cp.bincount(joint_idx, minlength=n_bins * n_bins).reshape(n_bins, n_bins)

    hx = gpu_entropy_from_counts(cx, n)
    hy = gpu_entropy_from_counts(cy, n)

    if hx == 0.0 and hy == 0.0:
        return 1.0
    if hx == 0.0 or hy == 0.0:
        return 0.0

    px = cx.astype(cp.float32) / cp.float32(n)
    py = cy.astype(cp.float32) / cp.float32(n)
    pxy = cxy.astype(cp.float32) / cp.float32(n)

    nz = pxy > 0
    pxy_nz = pxy[nz]

    px_py = px[:, None] * py[None, :]
    denom = px_py[nz]

    mi = cp.sum(pxy_nz * cp.log(pxy_nz / denom))
    nmi = mi / ((hx + hy) / 2.0)

    return float(nmi)


def quantile_discretize_gpu(X: cp.ndarray, n_bins: int) -> cp.ndarray:
    n_rows, n_cols = X.shape
    X_disc = cp.empty((n_rows, n_cols), dtype=cp.int32)

    quantiles = cp.linspace(0, 1, n_bins + 1, dtype=cp.float32)

    for j in tqdm(range(n_cols), desc=f"Quantile discretization (bins={n_bins})"):
        col = X[:, j]
        edges = cp.quantile(col, quantiles)
        edges = cp.unique(edges)

        if edges.size <= 2:
            X_disc[:, j] = 0
            continue

        internal_edges = edges[1:-1]
        X_disc[:, j] = cp.digitize(col, internal_edges, right=False).astype(cp.int32)

    return X_disc


def build_nmi_similarity_matrix_gpu_full(X_disc: cp.ndarray, feature_names, n_bins: int) -> pd.DataFrame:
    n_features = X_disc.shape[1]
    sim = cp.empty((n_features, n_features), dtype=cp.float32)

    sim.fill(cp.nan)
    for i in range(n_features):
        sim[i, i] = 1.0

    total_pairs = n_features * (n_features - 1) // 2

    with tqdm(total=total_pairs, desc=f"Computing pairwise NMI FULL (bins={n_bins})") as pbar:
        for i in range(n_features):
            xi = X_disc[:, i]
            for j in range(i + 1, n_features):
                xj = X_disc[:, j]
                val = gpu_nmi_discrete(xi, xj, n_bins=n_bins)
                sim[i, j] = val
                sim[j, i] = val
                pbar.update(1)

    sim_cpu = cp.asnumpy(sim)
    return pd.DataFrame(sim_cpu, index=feature_names, columns=feature_names)


def build_nmi_similarity_matrix_gpu_relative_lag(
    X_disc: cp.ndarray,
    feature_names,
    base_feature_names,
    max_lag: int,
    n_bins: int
) -> pd.DataFrame:
    """
    Approximate NMI matrix using only:
      sim(base_i, base_j, delta_lag)
    and then reconstruct the full lagged feature-feature matrix.

    Assumption:
      similarity depends mainly on relative lag, not absolute lag index.
    """
    n_features = len(feature_names)
    feature_to_idx = {name: i for i, name in enumerate(feature_names)}

    # Parse metadata once
    meta = [parse_feature_name(f) for f in feature_names]

    # Precompute reduced map
    sim_map = {}

    total_reduced = len(base_feature_names) * len(base_feature_names) * (2 * max_lag + 1)

    with tqdm(total=total_reduced, desc=f"Computing reduced NMI map (bins={n_bins})") as pbar:
        for base_i in base_feature_names:
            for base_j in base_feature_names:
                for delta in range(-max_lag, max_lag + 1):
                    # canonical representative:
                    # if delta >= 0 -> compare base_i_lag_delta vs base_j_lag_0
                    # if delta < 0  -> compare base_i_lag_0 vs base_j_lag_-delta
                    lag_i = max(delta, 0)
                    lag_j = max(-delta, 0)

                    fi = f"{base_i}_lag_{lag_i}"
                    fj = f"{base_j}_lag_{lag_j}"

                    idx_i = feature_to_idx[fi]
                    idx_j = feature_to_idx[fj]

                    val = gpu_nmi_discrete(X_disc[:, idx_i], X_disc[:, idx_j], n_bins=n_bins)
                    sim_map[(base_i, base_j, delta)] = val
                    pbar.update(1)

    # Reconstruct full matrix
    sim = cp.empty((n_features, n_features), dtype=cp.float32)
    sim.fill(cp.nan)

    total_pairs = n_features * (n_features + 1) // 2
    with tqdm(total=total_pairs, desc=f"Reconstructing full NMI matrix (bins={n_bins})") as pbar:
        for i in range(n_features):
            side_i, level_i, lag_i, base_i = meta[i]
            sim[i, i] = 1.0
            pbar.update(1)

            for j in range(i + 1, n_features):
                side_j, level_j, lag_j, base_j = meta[j]
                delta = lag_i - lag_j
                val = sim_map[(base_i, base_j, delta)]
                sim[i, j] = val
                sim[j, i] = val
                pbar.update(1)

    sim_cpu = cp.asnumpy(sim)
    return pd.DataFrame(sim_cpu, index=feature_names, columns=feature_names)


# ============================================================
# UTILS - SPEARMAN / CORRELATION
# ============================================================

def rank_columns_gpu(X: cp.ndarray) -> cp.ndarray:
    n_rows, n_cols = X.shape
    ranks = cp.empty_like(X, dtype=cp.float32)

    for j in tqdm(range(n_cols), desc="Ranking columns for Spearman"):
        order = cp.argsort(X[:, j])
        ranks[order, j] = cp.arange(n_rows, dtype=cp.float32)

    return ranks


def corrcoef_gpu_to_pandas(X: cp.ndarray, feature_names) -> pd.DataFrame:
    corr = cp.corrcoef(X, rowvar=False)
    corr_cpu = cp.asnumpy(corr)
    return pd.DataFrame(corr_cpu, index=feature_names, columns=feature_names)


# ============================================================
# UTILS - MEMORY-EFFICIENT LAGGED MATRIX
# ============================================================

def build_lagged_cudf_ordered_memory_efficient(df_base: cudf.DataFrame, max_lag: int) -> cudf.DataFrame:
    n = len(df_base)
    if max_lag >= n:
        raise ValueError(f"max_lag={max_lag} must be smaller than number of rows={n}")

    ordered_series = []

    for base_col in df_base.columns:
        for lag in range(max_lag + 1):
            start = max_lag - lag
            stop = n - lag
            col = df_base[base_col].iloc[start:stop].reset_index(drop=True)
            col.name = f"{base_col}_lag_{lag}"
            ordered_series.append(col)

    df_lagged = cudf.concat(ordered_series, axis=1)
    return df_lagged


# ============================================================
# LOAD CSV ON GPU
# ============================================================

print("Loading CSV on GPU with cuDF...")

if NROWS is not None:
    df_10 = cudf.read_csv(CSV_PATH, header=None, nrows=NROWS)
else:
    df_10 = cudf.read_csv(CSV_PATH, header=None)

print(f"Loaded shape: {df_10.shape}")


# ============================================================
# EXTRACT ASK/BID SIZE COLUMNS
# ============================================================

print("Extracting size columns...")

ask_size_indices = [4 * i + 1 for i in range(NUM_LEVELS)]
bid_size_indices = [4 * i + 3 for i in range(NUM_LEVELS)]

selected_indices = []
selected_names = []

for i in range(NUM_LEVELS):
    selected_indices.append(ask_size_indices[i])
    selected_names.append(f"ask_{i}")

    selected_indices.append(bid_size_indices[i])
    selected_names.append(f"bid_{i}")

df_sizes = df_10.iloc[:, selected_indices]
df_sizes.columns = selected_names

print(f"df_sizes shape: {df_sizes.shape}")
print("Base feature order:")
print(selected_names)


# ============================================================
# CORE ANALYSIS FOR A GIVEN LAG
# ============================================================

def run_full_analysis_for_lag(df_sizes: cudf.DataFrame, max_lag: int, n_bins_list):
    print("\n" + "=" * 70)
    print(f"RUNNING ANALYSIS FOR MAX_LAG = {max_lag}")
    print("=" * 70)

    n_rows = len(df_sizes)
    if max_lag >= n_rows:
        raise ValueError(f"max_lag={max_lag} must be smaller than number of rows={n_rows}")

    expected_n_features = 2 * NUM_LEVELS * (max_lag + 1)
    if expected_n_features > MAX_FEATURES_HARD_LIMIT:
        raise MemoryError(
            f"Expected features = {expected_n_features}, which exceeds the hard safety limit "
            f"({MAX_FEATURES_HARD_LIMIT}). Reduce max_lag."
        )

    print("Building lagged dataframe (memory-efficient)...")
    df_lagged = build_lagged_cudf_ordered_memory_efficient(df_sizes, max_lag)
    feature_names = list(df_lagged.columns)
    base_feature_names = list(df_sizes.columns)

    print(f"df_lagged shape: {df_lagged.shape}")
    print(f"Number of features: {len(feature_names)}")
    print("First 12 feature names:")
    print(feature_names[:12])
    print(f"Expected number of features: {expected_n_features}")

    print("Converting lagged matrix to CuPy array...")
    dtype = cp.float32 if USE_FLOAT32 else cp.float64
    X = df_lagged.to_cupy().astype(dtype, copy=False)

    n_samples, n_features = X.shape
    print(f"X shape: {X.shape}")

    del df_lagged
    clear_gpu_memory()

    # --------------------------------------------------------
    # PEARSON
    # --------------------------------------------------------
    print("Computing Pearson correlation matrix on GPU...")
    pearson_df = corrcoef_gpu_to_pandas(X, feature_names)
    print("Pearson done.")
    quick_matrix_checks(pearson_df, "Pearson")

    # --------------------------------------------------------
    # SPEARMAN
    # --------------------------------------------------------
    print("Computing Spearman correlation matrix on GPU...")
    X_rank = rank_columns_gpu(X)
    spearman_df = corrcoef_gpu_to_pandas(X_rank, feature_names)
    print("Spearman done.")
    quick_matrix_checks(spearman_df, "Spearman")

    del X_rank
    clear_gpu_memory()

    # --------------------------------------------------------
    # NMI FOR MULTIPLE BIN COUNTS
    # --------------------------------------------------------
    nmi_results = {}

    if n_features <= MAX_FEATURES_FOR_NMI:
        for n_bins in n_bins_list:
            print("\n" + "-" * 60)
            print(f"Running NMI with n_bins = {n_bins}")
            print("-" * 60)

            print("Discretizing for NMI on GPU...")
            X_disc = quantile_discretize_gpu(X, n_bins=n_bins)

            if USE_RELATIVE_LAG_NMI:
                print("Computing NMI similarity matrix on GPU (relative-lag approximation)...")
                nmi_df = build_nmi_similarity_matrix_gpu_relative_lag(
                    X_disc=X_disc,
                    feature_names=feature_names,
                    base_feature_names=base_feature_names,
                    max_lag=max_lag,
                    n_bins=n_bins
                )
            else:
                print("Computing NMI similarity matrix on GPU (full pairwise)...")
                nmi_df = build_nmi_similarity_matrix_gpu_full(
                    X_disc=X_disc,
                    feature_names=feature_names,
                    n_bins=n_bins
                )

            print(f"NMI done for bins={n_bins}.")
            quick_matrix_checks(nmi_df, f"NMI (bins={n_bins})")
            print(f"NMI mean (off-diagonal): {offdiag_mean(nmi_df):.4f}")

            if SAVE_CSV:
                suffix = "_rellag" if USE_RELATIVE_LAG_NMI else ""
                nmi_path = f"lob_similarity_nmi{suffix}_lag{max_lag}_bins{n_bins}.csv"
                nmi_df.to_csv(nmi_path, index=True)
                print(f"Saved NMI -> {os.path.abspath(nmi_path)}")

            if PLOT_HEATMAPS and n_features <= MAX_FEATURES_FOR_PLOT:
                print(f"Plotting NMI heatmap for bins={n_bins}...")
                plot_heatmap(
                    nmi_df,
                    f"LOB NMI Map (lag={max_lag}, bins={n_bins})",
                    cmap="YlGnBu",
                    vmin=0,
                    vmax=1
                )

            nmi_results[n_bins] = nmi_df

            del X_disc
            clear_gpu_memory()
    else:
        print(
            f"Skipping all NMI runs: number of features ({n_features}) exceeds "
            f"MAX_FEATURES_FOR_NMI ({MAX_FEATURES_FOR_NMI})."
        )

    del X
    clear_gpu_memory()

    # --------------------------------------------------------
    # SAVE PEARSON / SPEARMAN
    # --------------------------------------------------------
    if SAVE_CSV:
        pearson_path = f"lob_similarity_pearson_lag{max_lag}.csv"
        spearman_path = f"lob_similarity_spearman_lag{max_lag}.csv"

        pearson_df.to_csv(pearson_path, index=True)
        spearman_df.to_csv(spearman_path, index=True)

        print("\nSaved files:")
        print(f"Pearson  -> {os.path.abspath(pearson_path)}")
        print(f"Spearman -> {os.path.abspath(spearman_path)}")

    # --------------------------------------------------------
    # PLOT PEARSON / SPEARMAN
    # --------------------------------------------------------
    if PLOT_HEATMAPS:
        if n_features <= MAX_FEATURES_FOR_PLOT:
            print("\nPlotting Pearson...")
            plot_heatmap(
                pearson_df,
                f"LOB Pearson Correlation Map (lag={max_lag})",
                cmap="coolwarm",
                vmin=-1,
                vmax=1
            )

            print("Plotting Spearman...")
            plot_heatmap(
                spearman_df,
                f"LOB Spearman Correlation Map (lag={max_lag})",
                cmap="coolwarm",
                vmin=-1,
                vmax=1
            )
        else:
            print(
                f"Skipping plots: number of features ({n_features}) exceeds "
                f"MAX_FEATURES_FOR_PLOT ({MAX_FEATURES_FOR_PLOT})."
            )

    # --------------------------------------------------------
    # QUICK SUMMARY
    # --------------------------------------------------------
    print("\n--- QUICK STATS ---")
    print(f"Pearson mean (off-diagonal) : {offdiag_mean(pearson_df):.4f}")
    print(f"Spearman mean (off-diagonal): {offdiag_mean(spearman_df):.4f}")
    for n_bins, nmi_df in nmi_results.items():
        print(f"NMI mean (off-diagonal) [bins={n_bins}] : {offdiag_mean(nmi_df):.4f}")

    return {
        "pearson": pearson_df,
        "spearman": spearman_df,
        "nmi_by_bins": nmi_results,
        "feature_names": feature_names
    }


# ============================================================
# RUN MULTI-LAG ANALYSIS
# ============================================================

results = {}

for lag in LAG_LIST:
    try:
        results[lag] = run_full_analysis_for_lag(df_sizes, lag, N_BINS_LIST)
    except Exception as e:
        print(f"\n[ERROR] Failed for lag={lag}: {type(e).__name__}: {e}")
        results[lag] = {
            "pearson": None,
            "spearman": None,
            "nmi_by_bins": {},
            "feature_names": None,
            "error": str(e)
        }
    finally:
        clear_gpu_memory()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("MULTI-LAG + MULTI-BIN ANALYSIS COMPLETED")
print("=" * 70)

for lag in LAG_LIST:
    print(f"\nMAX_LAG = {lag}")

    if results[lag]["pearson"] is None:
        print(f"  Failed: {results[lag].get('error', 'unknown error')}")
        continue

    pearson_df = results[lag]["pearson"]
    spearman_df = results[lag]["spearman"]
    nmi_by_bins = results[lag]["nmi_by_bins"]

    print(f"  Pearson off-diagonal mean : {offdiag_mean(pearson_df):.4f}")
    print(f"  Spearman off-diagonal mean: {offdiag_mean(spearman_df):.4f}")

    if len(nmi_by_bins) == 0:
        print("  NMI off-diagonal mean     : skipped")
    else:
        for n_bins in N_BINS_LIST:
            if n_bins in nmi_by_bins:
                print(f"  NMI off-diagonal mean [bins={n_bins}] : {offdiag_mean(nmi_by_bins[n_bins]):.4f}")

In [ ]:
# ============================================================
# POST-NMI ANALYSIS (USING RESULTS DICT)
# ============================================================

def compute_extra_stats(sim_df: pd.DataFrame) -> pd.DataFrame:
    feature_names = list(sim_df.columns)
    n = len(feature_names)

    meta = [parse_feature_name(f) for f in feature_names]

    values = []
    same_level_vals = []
    diff_level_vals = []

    for i in range(n):
        for j in range(i + 1, n):
            val = sim_df.iloc[i, j]
            values.append(val)

            side_i, level_i, lag_i, base_i = meta[i]
            side_j, level_j, lag_j, base_j = meta[j]

            if level_i == level_j:
                same_level_vals.append(val)
            else:
                diff_level_vals.append(val)

    values = np.array(values)
    same_level_vals = np.array(same_level_vals)
    diff_level_vals = np.array(diff_level_vals)

    unique_vals, counts = np.unique(np.round(values, 6), return_counts=True)

    df = pd.DataFrame({
        "metric": [
            "n_pairs",
            "n_unique_values (rounded)",
            "n_repeated_values",
            "mean_total",
            "mean_same_level",
            "mean_diff_level",
            "max_total",
            "min_total",
            "max_same_level",
            "min_same_level",
            "max_diff_level",
            "min_diff_level",
        ],
        "value": [
            len(values),
            len(unique_vals),
            int(np.sum(counts > 1)),
            np.nanmean(values),
            np.nanmean(same_level_vals),
            np.nanmean(diff_level_vals),
            np.nanmax(values),
            np.nanmin(values),
            np.nanmax(same_level_vals),
            np.nanmin(same_level_vals),
            np.nanmax(diff_level_vals),
            np.nanmin(diff_level_vals),
        ]
    })

    return df


# ============================================================
# RUN ANALYSIS ON ALL RESULTS
# ============================================================

for lag in results:
    res = results[lag]

    if res["nmi_by_bins"] is None or len(res["nmi_by_bins"]) == 0:
        print(f"\n[SKIP] lag={lag} → no NMI computed")
        continue

    for n_bins, nmi_df in res["nmi_by_bins"].items():
        print("\n" + "="*60)
        print(f"NMI EXTRA ANALYSIS | lag={lag} | bins={n_bins}")
        print("="*60)

        extra_stats_df = compute_extra_stats(nmi_df)

        display(
            extra_stats_df.style
            .format("{:.6f}", subset=["value"])
            .set_caption(f"Extra Stats | lag={lag}, bins={n_bins}")
        )